# M02a · Intégrales et densités — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/m02a_integrales_densites/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/m02a_integrales_densites/03_notebook.ipynb`, créée par `python tools/start_chapter.py M02a`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | M02a.1 à M02a.10, M02a.13 | Les exercices papier (vérification) | ✏️ 🔗 | | |
| A | M02a.16 | Rectangles et trapèzes · `riemann_sum`, `trapezoid` | 🔨 | ★ | 20 |
|  | M02a.17 | La méthode de Simpson · `simpson` | 🔨 | ★★ | 15 |
|  | M02a.18 | De la densité à la fonction de répartition, puis au quantile par dichotomie · `cdf_from_pdf`, `quantile` | 🔨 | ★★ | 25 |
|  | M02a.19 | Vérifier avec `scipy.integrate.quad` et SymPy | 📦 | ★ | 15 |
| B | M02a.20 | Prédire : le pas divisé par 2, l'erreur divisée par combien ? | 🔮 | ★ | 10 |
|  | M02a.21 | L'intégrale de Gauss par le calcul : la borne qui s'éloigne | 🔬 | ★★ | 20 |
|  | M02a.22 | L'aire qui se remplit : la courbe, les rectangles et la primitive | 🎨 | ★ | 20 |
| C | M02a.23 | Chasse à l'erreur : bornes inversées, primitive fausse, aire négative | 🐛 | ★ | 15 |
|  | M02a.24 | Vérifier une primitive avec SymPy : dériver le résultat | 🛠️ | ★ | 10 |
|  | M02a.25 | Défi : la meilleure précision avec dix évaluations de la fonction | 🏆 | ★★★ | 30 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M02a")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : sommes de Riemann, primitives, intégrales et aires, intégration par parties et changement de variable, intégrales impropres, trapèzes et Simpson, densités, probabilités, fonctions de répartition et quantiles, AUC.
- Écrire les cinq fonctions de `mymath/integrate.py` et les tester, puis comparer avec SciPy et SymPy.
- Prédire l'effet d'un pas divisé par 2, approcher l'intégrale de Gauss, dessiner l'aire qui se remplit, chasser des erreurs de calcul intégral, vérifier une primitive, relever le défi des dix évaluations.

**Rappel express.** $\int_a^b f = F(b) - F(a)$ avec $F' = f$. $h = \frac{b - a}{n}$ ; trapèzes : $h\big(\frac{f_0}{2} + f_1 + \cdots + \frac{f_n}{2}\big)$ ; Simpson ($n$ pair) : $\frac{h}{3}(f_0 + 4f_1 + 2f_2 + \cdots + 4f_{n-1} + f_n)$. Densité : $f \ge 0$, $\int f = 1$, $P(a \le X \le b) = \int_a^b f$ ; $F(x) = P(X \le x)$ ; quantile $q_p$ : $F(q_p) = p$.

## Partie 0 · Vérifier tes exercices papier (✏️ M02a.1 à M02a.10, 🔗 M02a.13)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un entier, ou un nombre avec un point décimal, `-1.25`, arrondi comme le demande la question ;
- une fraction irréductible entre guillemets, `"5/9"` ;
- une valeur exacte entre guillemets, avec `sqrt`, `pi`, `exp` ou `ln`, `"sqrt(5)/3"` ;
- une expression en $x$ entre guillemets, avec `**` pour les puissances et des parenthèses autour d'un dénominateur ou d'un exposant, `"x**4/4 - 3/(2*x)"`, `"exp(3*x)"` (une primitive se vérifie à une constante près) ;
- une liste `[-1.25, 1]`, un intervalle `"]-oo ; -7]"`, une lettre `"C"`, ou `True` / `False`.

Les réponses pas encore remplies affichent ⏳.

In [ ]:
from fractions import Fraction   # Fraction(5, 9) is accepted too, as "5/9"
import sympy as sp
x = sp.Symbol("x")                  # an expression typed without quotes works too

**Ex M02a.1 — Sommes de Riemann à la main : rectangles à gauche, à droite, au milieu**

In [ ]:
# Ex M02a.1: the values you found on paper
answer_M02a_1a = ...  # a) a number (1 decimal)
answer_M02a_1b = ...  # b) a number (2 decimals)
answer_M02a_1c = ...  # c) a number (2 decimals)
answer_M02a_1d = ...  # d) a number (3 decimals)
answer_M02a_1e = ...  # e) a number
answer_M02a_1f = ...  # f) a number (4 decimals)
answer_M02a_1g = ...  # g) an integer
answer_M02a_1h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_M02a_1a, answer_M02a_1b, answer_M02a_1c, answer_M02a_1d, answer_M02a_1e, answer_M02a_1f, answer_M02a_1g, answer_M02a_1h]):
    wbm.check(f"M02a.1{letter}", answer)

**Ex M02a.2 — Primitives usuelles**

In [ ]:
# Ex M02a.2: the values you found on paper
answer_M02a_2a = ...  # a) an expression in x, as text
answer_M02a_2b = ...  # b) an expression in x, as text
answer_M02a_2c = ...  # c) an expression in x, as text
answer_M02a_2d = ...  # d) an expression in x, as text
answer_M02a_2e = ...  # e) an expression in x, as text
answer_M02a_2f = ...  # f) an expression in x, as text
answer_M02a_2g = ...  # g) an expression in x, as text (with its constant)
answer_M02a_2h = ...  # h) an expression in x, as text

for letter, answer in zip("abcdefgh", [answer_M02a_2a, answer_M02a_2b, answer_M02a_2c, answer_M02a_2d, answer_M02a_2e, answer_M02a_2f, answer_M02a_2g, answer_M02a_2h]):
    wbm.check(f"M02a.2{letter}", answer)

**Ex M02a.3 — Intégrales et aires : le signe, l'aire entre deux courbes**

In [ ]:
# Ex M02a.3: the values you found on paper
answer_M02a_3a = ...  # a) an integer
answer_M02a_3b = ...  # b) an integer
answer_M02a_3c = ...  # c) an integer
answer_M02a_3d = ...  # d) an integer
answer_M02a_3e = ...  # e) an irreducible fraction, as text
answer_M02a_3f = ...  # f) a list, in increasing order
answer_M02a_3g = ...  # g) an irreducible fraction, as text
answer_M02a_3h = ...  # h) an irreducible fraction, as text

for letter, answer in zip("abcdefgh", [answer_M02a_3a, answer_M02a_3b, answer_M02a_3c, answer_M02a_3d, answer_M02a_3e, answer_M02a_3f, answer_M02a_3g, answer_M02a_3h]):
    wbm.check(f"M02a.3{letter}", answer)

**Ex M02a.4 — Intégration par parties**

In [ ]:
# Ex M02a.4: the values you found on paper
answer_M02a_4a = ...  # a) 1 or 2
answer_M02a_4b = ...  # b) a number (4 decimals)
answer_M02a_4c = ...  # c) an expression in x, as text
answer_M02a_4d = ...  # d) an integer
answer_M02a_4e = ...  # e) an expression in x, as text
answer_M02a_4f = ...  # f) a number (4 decimals)

for letter, answer in zip("abcdef", [answer_M02a_4a, answer_M02a_4b, answer_M02a_4c, answer_M02a_4d, answer_M02a_4e, answer_M02a_4f]):
    wbm.check(f"M02a.4{letter}", answer)

**Ex M02a.5 — Changement de variable**

In [ ]:
# Ex M02a.5: the values you found on paper
answer_M02a_5a = ...  # a) a list
answer_M02a_5b = ...  # b) an irreducible fraction, as text
answer_M02a_5c = ...  # c) an expression in x, as text
answer_M02a_5d = ...  # d) a number (4 decimals)
answer_M02a_5e = ...  # e) a number (4 decimals)
answer_M02a_5f = ...  # f) an irreducible fraction, as text

for letter, answer in zip("abcdef", [answer_M02a_5a, answer_M02a_5b, answer_M02a_5c, answer_M02a_5d, answer_M02a_5e, answer_M02a_5f]):
    wbm.check(f"M02a.5{letter}", answer)

**Ex M02a.6 — Intégrales impropres : converge, diverge, valeur**

In [ ]:
# Ex M02a.6: the values you found on paper
answer_M02a_6a = ...  # a) an irreducible fraction, as text
answer_M02a_6b = ...  # b) an integer
answer_M02a_6c = ...  # c) an integer
answer_M02a_6d = ...  # d) True or False
answer_M02a_6e = ...  # e) an irreducible fraction, as text
answer_M02a_6f = ...  # f) an interval, as text
answer_M02a_6g = ...  # g) an irreducible fraction, as text
answer_M02a_6h = ...  # h) an exact value, as text

for letter, answer in zip("abcdefgh", [answer_M02a_6a, answer_M02a_6b, answer_M02a_6c, answer_M02a_6d, answer_M02a_6e, answer_M02a_6f, answer_M02a_6g, answer_M02a_6h]):
    wbm.check(f"M02a.6{letter}", answer)

**Ex M02a.7 — Trapèzes et Simpson à la main, et leurs erreurs**

In [ ]:
# Ex M02a.7: the values you found on paper
answer_M02a_7a = ...  # a) a number (1 decimal)
answer_M02a_7b = ...  # b) an irreducible fraction, as text
answer_M02a_7c = ...  # c) a number (4 decimals)
answer_M02a_7d = ...  # d) a number (4 decimals)
answer_M02a_7e = ...  # e) a number (1 decimal)
answer_M02a_7f = ...  # f) a number (5 decimals)
answer_M02a_7g = ...  # g) an integer
answer_M02a_7h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_M02a_7a, answer_M02a_7b, answer_M02a_7c, answer_M02a_7d, answer_M02a_7e, answer_M02a_7f, answer_M02a_7g, answer_M02a_7h]):
    wbm.check(f"M02a.7{letter}", answer)

**Ex M02a.8 — Vérifier qu'une fonction est une densité, trouver sa constante**

In [ ]:
# Ex M02a.8: the values you found on paper
answer_M02a_8a = ...  # a) an irreducible fraction, as text
answer_M02a_8b = ...  # b) an integer
answer_M02a_8c = ...  # c) an integer
answer_M02a_8d = ...  # d) an irreducible fraction, as text
answer_M02a_8e = ...  # e) True or False
answer_M02a_8f = ...  # f) True or False
answer_M02a_8g = ...  # g) an irreducible fraction, as text
answer_M02a_8h = ...  # h) an integer

for letter, answer in zip("abcdefgh", [answer_M02a_8a, answer_M02a_8b, answer_M02a_8c, answer_M02a_8d, answer_M02a_8e, answer_M02a_8f, answer_M02a_8g, answer_M02a_8h]):
    wbm.check(f"M02a.8{letter}", answer)

**Ex M02a.9 — La probabilité d'un intervalle, et la valeur moyenne d'une fonction**

In [ ]:
# Ex M02a.9: the values you found on paper
answer_M02a_9a = ...  # a) an irreducible fraction, as text
answer_M02a_9b = ...  # b) an irreducible fraction, as text
answer_M02a_9c = ...  # c) a number
answer_M02a_9d = ...  # d) an irreducible fraction, as text
answer_M02a_9e = ...  # e) a number (4 decimals)
answer_M02a_9f = ...  # f) a number (4 decimals)
answer_M02a_9g = ...  # g) an integer
answer_M02a_9h = ...  # h) a number (4 decimals)

for letter, answer in zip("abcdefgh", [answer_M02a_9a, answer_M02a_9b, answer_M02a_9c, answer_M02a_9d, answer_M02a_9e, answer_M02a_9f, answer_M02a_9g, answer_M02a_9h]):
    wbm.check(f"M02a.9{letter}", answer)

**Ex M02a.10 — Fonction de répartition, médiane et percentiles d'une loi donnée**

In [ ]:
# Ex M02a.10: the values you found on paper
answer_M02a_10a = ...  # a) an irreducible fraction, as text
answer_M02a_10b = ...  # b) an expression in x, as text
answer_M02a_10c = ...  # c) an exact value, as text
answer_M02a_10d = ...  # d) an integer
answer_M02a_10e = ...  # e) an irreducible fraction, as text
answer_M02a_10f = ...  # f) a number (4 decimals)
answer_M02a_10g = ...  # g) a number (4 decimals)
answer_M02a_10h = ...  # h) a number (4 decimals)

for letter, answer in zip("abcdefgh", [answer_M02a_10a, answer_M02a_10b, answer_M02a_10c, answer_M02a_10d, answer_M02a_10e, answer_M02a_10f, answer_M02a_10g, answer_M02a_10h]):
    wbm.check(f"M02a.10{letter}", answer)

**Ex M02a.13 — L'aire sous une courbe ROC donnée par une formule**

In [ ]:
# Ex M02a.13: the values you found on paper
answer_M02a_13a = ...  # a) an irreducible fraction, as text
answer_M02a_13b = ...  # b) an irreducible fraction, as text
answer_M02a_13c = ...  # c) a letter, as text
answer_M02a_13d = ...  # d) a number (3 decimals)
answer_M02a_13e = ...  # e) a number (4 decimals)
answer_M02a_13f = ...  # f) True or False
answer_M02a_13g = ...  # g) a number (4 decimals)
answer_M02a_13h = ...  # h) an irreducible fraction, as text

for letter, answer in zip("abcdefgh", [answer_M02a_13a, answer_M02a_13b, answer_M02a_13c, answer_M02a_13d, answer_M02a_13e, answer_M02a_13f, answer_M02a_13g, answer_M02a_13h]):
    wbm.check(f"M02a.13{letter}", answer)

## Partie A · Ta librairie : sommes de Riemann, trapèzes, Simpson, fonction de répartition et quantile

Fiche §1, §2, §5 et §7. Les cinq fonctions de `mymath/integrate.py`, puis les mêmes calculs avec SciPy et SymPy.

In [ ]:
# Tools of the notebook (parts A to C)
import math

import matplotlib.pyplot as plt
import numpy as np
import sympy as sp
from scipy.integrate import quad

from wbm.nbtools import checked, drawn, run_pytest_group, verdict


def raised(function, *args):
    """The name of the exception that function(*args) raises, or "aucune" if it returns.

    NotImplementedError (a TODO not done yet) goes through, for wbm.attempt to show ⏳."""
    try:
        function(*args)
    except NotImplementedError:
        raise
    except Exception as exc:  # noqa: BLE001 - any other error is the answer
        return type(exc).__name__
    return "aucune"

### Ex M02a.16 — Rectangles et trapèzes · `riemann_sum`, `trapezoid` 🔨 ★ ⏱️ 20 min
**Objectif :** programmer les trois sommes de Riemann et la méthode des trapèzes, avec leurs cas d'erreur.  
**Prérequis :** fiche §1 et §5 · **mymath :** `integrate.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme dans les chapitres précédents). Ouvre ta copie `mon_travail/mymath/integrate.py`, lis la docstring de la fonction en entier, remplace la ligne `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. **Nouveau dans ce chapitre**, le premier de la partie I de workbookMaths : NumPy est permis dans `mymath` (importe-le en haut du fichier si tu t'en sers). Mais ni `scipy.integrate` ni `np.trapezoid` : les tests comparent tes résultats à eux, et un test vérifie que tu ne les appelles pas. Écris toi-même les sommes : c'est l'exercice. Chaque fonction renvoie un `float` de Python (`float(...)` convertit un nombre de NumPy).

Écris `riemann_sum(f, a, b, n, rule="left")` puis `trapezoid(f, a, b, n)` dans `mon_travail/mymath/integrate.py` (fiche §1 et §5). Les deux découpent $[a ; b]$ en $n$ intervalles de largeur $h = \frac{b - a}{n}$ ; `rule` vaut `"left"`, `"right"` ou `"midpoint"`. Elles lèvent une `ValueError` si $n < 1$, et `riemann_sum` aussi pour une règle inconnue.

La vérification calcule :
- a) `riemann_sum(lambda x: x ** 3, 1, 3, 4)`, b) la même avec `rule="right"`, c) avec `rule="midpoint"` ;
- d) `trapezoid(math.exp, 0, 1, 10)` (4 décimales) ;
- e) le nom de l'erreur levée par `riemann_sum(math.exp, 0, 1, 0)` (avec `raised`, de la cellule des outils) ;
- f) le nom de l'erreur levée par `riemann_sum(math.exp, 0, 1, 4, "center")` ;
- g) le type de `riemann_sum(lambda x: 2, 0, 1, 3)`, en texte (`"float"` pour un flottant) ;
- h) `trapezoid(lambda x: x ** 3, 1, 3, 4)`.

Puis, dans ta copie : compare h) à a) et b). Quelle relation entre les trapèzes et les deux sommes de Riemann le résultat montre-t-il ? L'intégrale exacte vaut 20 : laquelle des quatre valeurs en est la plus proche ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M02a")  # reload your saved file

In [ ]:
with checked("M02a.16a"):
    wbm.check("M02a.16a", mymath.integrate.riemann_sum(lambda x: x ** 3, 1, 3, 4), computed=True)
with checked("M02a.16b"):
    wbm.check("M02a.16b", mymath.integrate.riemann_sum(lambda x: x ** 3, 1, 3, 4, rule="right"), computed=True)
with checked("M02a.16c"):
    wbm.check("M02a.16c", mymath.integrate.riemann_sum(lambda x: x ** 3, 1, 3, 4, rule="midpoint"), computed=True)
with checked("M02a.16d"):
    wbm.check("M02a.16d", mymath.integrate.trapezoid(math.exp, 0, 1, 10), computed=True)
with checked("M02a.16e"):
    wbm.check("M02a.16e", raised(mymath.integrate.riemann_sum, math.exp, 0, 1, 0), computed=True)
with checked("M02a.16f"):
    wbm.check("M02a.16f", raised(mymath.integrate.riemann_sum, math.exp, 0, 1, 4, "center"), computed=True)
with checked("M02a.16g"):
    wbm.check("M02a.16g", type(mymath.integrate.riemann_sum(lambda x: 2, 0, 1, 3)).__name__, computed=True)
with checked("M02a.16h"):
    wbm.check("M02a.16h", mymath.integrate.trapezoid(lambda x: x ** 3, 1, 3, 4), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m02a_integrate.py", "test_riemann_sum_ or test_trapezoid_ or test_integrate_uses")

### Ex M02a.17 — La méthode de Simpson · `simpson` 🔨 ★★ ⏱️ 15 min
**Objectif :** programmer la méthode de Simpson, et vérifier son exactitude et son ordre.  
**Prérequis :** fiche §5 ; Ex M02a.16 · **mymath :** `integrate.py` · **Parcours :** C

Écris `simpson(f, a, b, n)` (fiche §5) : poids $1, 4, 2, 4, \ldots, 2, 4, 1$, le tout multiplié par $\frac{h}{3}$. Elle lève une `ValueError` si $n$ est impair ou plus petit que 2.

La vérification calcule :
- a) `simpson(math.exp, 0, 1, 4)` (6 décimales), à comparer avec $e - 1 \approx 1{,}718282$ ;
- b) le nom de l'erreur levée par `simpson(math.exp, 0, 1, 3)` ;
- c) `simpson(lambda x: x ** 3 - 2 * x, -1, 2, 2)` (4 décimales) ;
- d) le rapport des erreurs de `simpson(math.sin, 0, math.pi, n)` pour $n = 4$ et $n = 8$ (l'intégrale exacte vaut 2 ; 1 décimale) ;
- e) le type de `simpson(lambda x: 1, 0, 2, 2)`, en texte.

Dans ta copie : calcule c) à la main avec une primitive. Pourquoi Simpson tombe-t-il juste avec seulement deux intervalles ? Le rapport de d) est-il proche de la valeur annoncée par la fiche ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M02a")  # reload your saved file

In [ ]:
with checked("M02a.17a"):
    wbm.check("M02a.17a", mymath.integrate.simpson(math.exp, 0, 1, 4), computed=True)
with checked("M02a.17b"):
    wbm.check("M02a.17b", raised(mymath.integrate.simpson, math.exp, 0, 1, 3), computed=True)
with checked("M02a.17c"):
    wbm.check("M02a.17c", mymath.integrate.simpson(lambda x: x ** 3 - 2 * x, -1, 2, 2), computed=True)
with checked("M02a.17d"):
    error_4 = abs(mymath.integrate.simpson(math.sin, 0, math.pi, 4) - 2)
    error_8 = abs(mymath.integrate.simpson(math.sin, 0, math.pi, 8) - 2)
    wbm.check("M02a.17d", error_4 / error_8, computed=True)
with checked("M02a.17e"):
    wbm.check("M02a.17e", type(mymath.integrate.simpson(lambda x: 1, 0, 2, 2)).__name__, computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m02a_integrate.py", "test_simpson_")

### Ex M02a.18 — De la densité à la fonction de répartition, puis au quantile par dichotomie · `cdf_from_pdf`, `quantile` 🔨 ★★ ⏱️ 25 min
**Objectif :** calculer une fonction de répartition par la méthode des trapèzes, puis un quantile par dichotomie.  
**Prérequis :** fiche §5 et §7 ; Ex M02a.16 ; R7 (dichotomie) · **mymath :** `integrate.py` · **Parcours :** C, E

Écris `cdf_from_pdf(pdf, x, lower, n=1000)`, qui calcule $F(x) = \int_{\text{lower}}^x \text{pdf}(t)\,dt$ par ta méthode des trapèzes (elle rend 0,0 si $x \le \text{lower}$), puis `quantile(cdf, p, low, high, tol=1e-10)`, qui trouve par dichotomie le plus petit $x$ de $[\text{low} ; \text{high}]$ tel que $\text{cdf}(x) \ge p$ (fiche §7 ; R7). Lis les docstrings : elles disent les cas d'erreur.

La cellule fournie définit la densité $6t(1 - t)$ sur $[0 ; 1]$ et sa fonction de répartition exacte. La vérification calcule :
- a) `cdf_from_pdf(pdf, 0.3, 0)` (3 décimales) ;
- b) `cdf_from_pdf(pdf, -0.5, 0)` ;
- c) `cdf_from_pdf(pdf, 0.3, 0, n=1)`, avec un seul trapèze (3 décimales) ;
- d) `quantile(cdf_exact, 0.5, 0, 1)`, la médiane (6 décimales) ;
- e) le quantile d'ordre 0,9 de la loi de fonction de répartition $1 - e^{-2x}$ (pour $x \ge 0$), cherché dans $[0 ; 10]$ (6 décimales) ;
- f) le nom de l'erreur levée par `quantile(cdf_exact, 1.2, 0, 1)` ;
- g) le percentile 95 de la densité `pdf`, en ne te servant **que de la densité** : `quantile` appliquée à `lambda x: cdf_from_pdf(pdf, x, 0)` (3 décimales).

Dans ta copie : compare e) à la formule $-\frac{\ln(1 - p)}{\lambda}$ de la fiche. Pourquoi c) est-il loin de a) ?

In [ ]:
def pdf(t):
    """A density on [0, 1]: 6 t (1 − t), and 0 elsewhere."""
    return 6 * t * (1 - t) if 0 <= t <= 1 else 0.0


def cdf_exact(x):
    """Its CDF, written with a primitive: 3x² − 2x³ on [0, 1]."""
    if x <= 0:
        return 0.0
    if x >= 1:
        return 1.0
    return 3 * x ** 2 - 2 * x ** 3

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M02a")  # reload your saved file

In [ ]:
with checked("M02a.18a"):
    wbm.check("M02a.18a", mymath.integrate.cdf_from_pdf(pdf, 0.3, 0), computed=True)
with checked("M02a.18b"):
    wbm.check("M02a.18b", mymath.integrate.cdf_from_pdf(pdf, -0.5, 0), computed=True)
with checked("M02a.18c"):
    wbm.check("M02a.18c", mymath.integrate.cdf_from_pdf(pdf, 0.3, 0, n=1), computed=True)
with checked("M02a.18d"):
    wbm.check("M02a.18d", mymath.integrate.quantile(cdf_exact, 0.5, 0, 1), computed=True)
with checked("M02a.18e"):
    wbm.check("M02a.18e", mymath.integrate.quantile(lambda x: 1 - math.exp(-2 * x), 0.9, 0, 10), computed=True)
with checked("M02a.18f"):
    wbm.check("M02a.18f", raised(mymath.integrate.quantile, cdf_exact, 1.2, 0, 1), computed=True)
with checked("M02a.18g"):
    numerical_cdf = lambda x: mymath.integrate.cdf_from_pdf(pdf, x, 0)   # noqa: E731
    wbm.check("M02a.18g", mymath.integrate.quantile(numerical_cdf, 0.95, 0, 1), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m02a_integrate.py", "test_cdf_from_pdf_ or test_quantile_")

### Ex M02a.19 — Vérifier avec `scipy.integrate.quad` et SymPy 📦 ★ ⏱️ 15 min
**Objectif :** calculer des intégrales avec SciPy (numériquement) et SymPy (exactement), et lire ce que chacun rend.  
**Prérequis :** fiche §2, §4 et §5 · **Parcours :** C, E

Deux outils calculent des intégrales. `quad(f, a, b)`, de `scipy.integrate`, rend un couple `(valeur, estimation_de_l_erreur)` et accepte des bornes infinies (`-math.inf`, `math.inf`). `sp.integrate(expression, x)` rend une primitive exacte, et `sp.integrate(expression, (x, a, b))` une intégrale exacte (`sp.oo` pour $+\infty$).

Calcule :
- a) avec `quad`, $\int_{-\infty}^{+\infty} e^{-x^2}\,dx$ (la valeur, 6 décimales) ;
- b) un booléen : l'estimation d'erreur rendue par `quad` en a) est-elle inférieure à $10^{-6}$ ?
- c) avec SymPy, $\int_0^{+\infty} x\,e^{-x}\,dx$ (un entier) ;
- d) avec SymPy, $\int_0^1 \frac{4}{1 + x^2}\,dx$ (une valeur exacte : le résultat de SymPy tel quel, ou converti en texte avec `str(...)`) ;
- e) avec `quad`, $\int_0^1 \frac{dx}{\sqrt{x}}$ (6 décimales), une intégrale impropre : la fonction explose en 0 ;
- f) le nom de la fonction spéciale que SymPy utilise pour écrire une primitive de $e^{-x^2}$ (un texte de trois lettres : regarde `sp.integrate(sp.exp(-x ** 2), x)`).

Dans ta copie : la primitive de f) contredit-elle la fiche, qui dit que $e^{-x^2}$ n'a pas de primitive écrite avec les fonctions usuelles ?

In [ ]:
x = sp.Symbol("x")

In [ ]:
answer_M02a_19a = ...  # a) the value of quad(...)
answer_M02a_19b = ...  # b) True or False
answer_M02a_19c = ...  # c) sp.integrate(...)
answer_M02a_19d = ...  # d) sp.integrate(...), or its str(...)
answer_M02a_19e = ...  # e) the value of quad(...)
answer_M02a_19f = ...  # f) a name, as text

In [ ]:
for letter, answer in zip("abcdef", [answer_M02a_19a, answer_M02a_19b, answer_M02a_19c, answer_M02a_19d,
                                      answer_M02a_19e, answer_M02a_19f]):
    wbm.check(f"M02a.19{letter}", answer)

## Partie B · Une prédiction, une expérience et une figure

Fiche §1, §2, §4 et §5. Quand la règle « ÷ 4, ÷ 16 » s'applique, et quand elle ne s'applique pas ; l'intégrale de Gauss par le calcul ; l'aire qui se remplit.

### Ex M02a.20 — Prédire : le pas divisé par 2, l'erreur divisée par combien ? 🔮 ★ ⏱️ 10 min
**Objectif :** prévoir de combien l'erreur d'une méthode diminue quand le pas est divisé par 2, et voir quand la règle de la fiche ne s'applique plus.  
**Prérequis :** fiche §5 · **Parcours :** C, E

On passe de $n = 4$ à $n = 8$ intervalles (le pas est divisé par 2). Pour chaque cas, de combien l'erreur est-elle divisée ? Choisis : (1) moins de 4 ; (2) environ 4 ; (3) environ 16 ; (4) bien plus que 16.
- a) la méthode des trapèzes pour $\int_0^1 e^x\,dx$ ;
- b) la méthode de Simpson pour la même intégrale ;
- c) la méthode de Simpson pour $\int_0^1 \sqrt{x}\,dx$ (attention : la dérivée de $\sqrt{x}$ devient infinie en 0) ;
- d) la méthode des trapèzes pour $\int_0^{2\pi} e^{\cos x}\,dx$, une fonction **périodique** intégrée sur une période entière.

**Avant d'exécuter quoi que ce soit**, écris tes quatre prédictions, vérifie-les, puis lance l'expérience. Réponds par le **numéro du choix**, de 1 à 4, et non par le facteur : pour « environ 4 », écris 2.

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_M02a_20a = ...  # a) 1, 2, 3 or 4, written BEFORE running the experiment below
prediction_M02a_20b = ...  # b) 1, 2, 3 or 4
prediction_M02a_20c = ...  # c) 1, 2, 3 or 4
prediction_M02a_20d = ...  # d) 1, 2, 3 or 4

In [ ]:
for letter, prediction in zip("abcd", [prediction_M02a_20a, prediction_M02a_20b, prediction_M02a_20c,
                                      prediction_M02a_20d]):
    wbm.check(f"M02a.20{letter}", prediction)

**L'expérience.** Elle ne s'exécute qu'une fois tes quatre prédictions écrites.

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_M02a_20a, prediction_M02a_20b, prediction_M02a_20c, prediction_M02a_20d]):
    print("⏳ Écris d'abord tes quatre prédictions, puis exécute à nouveau.")
else:
    def trapezoids(f, a, b, n):
        xs = np.linspace(a, b, n + 1)
        ys = np.array([f(x) for x in xs])
        return (b - a) / n * (ys[0] / 2 + ys[1:-1].sum() + ys[-1] / 2)


    def simpsons(f, a, b, n):
        xs = np.linspace(a, b, n + 1)
        ys = np.array([f(x) for x in xs])
        return (b - a) / n / 3 * (ys[0] + ys[-1] + 4 * ys[1:-1:2].sum() + 2 * ys[2:-1:2].sum())


    cases = {
        "a) trapèzes, e^x sur [0 ; 1]": (trapezoids, math.exp, 0, 1, math.e - 1),
        "b) Simpson, e^x sur [0 ; 1]": (simpsons, math.exp, 0, 1, math.e - 1),
        "c) Simpson, √x sur [0 ; 1]": (simpsons, math.sqrt, 0, 1, 2 / 3),
        "d) trapèzes, e^(cos x) sur [0 ; 2π]": (trapezoids, lambda x: math.exp(math.cos(x)), 0, 2 * math.pi,
                                               quad(lambda x: math.exp(math.cos(x)), 0, 2 * math.pi, epsabs=1e-14)[0]),
    }
    for name, (method, f, a, b, exact) in cases.items():
        error_4, error_8 = abs(method(f, a, b, 4) - exact), abs(method(f, a, b, 8) - exact)
        print(f"{name:38} erreur n = 4 : {error_4:.1e}   n = 8 : {error_8:.1e}   rapport : {error_4 / error_8:.1f}")

**Après l'expérience**, dans ta copie : quelle hypothèse de la fiche (« pour une fonction régulière ») n'est pas remplie en c) ? Dessine $e^{\cos x}$ sur $[0 ; 2\pi]$ et les quatre trapèzes : pourquoi leurs erreurs se compensent-elles si bien en d) ?

### Ex M02a.21 — L'intégrale de Gauss par le calcul : la borne qui s'éloigne 🔬 ★★ ⏱️ 20 min
**Objectif :** vérifier numériquement la valeur de l'intégrale de Gauss en reculant les bornes, et mesurer à quelle vitesse la queue devient négligeable.  
**Prérequis :** fiche §4 et §5 ; Ex M02a.19 · **Parcours :** C

On admet $\int_{-\infty}^{+\infty} e^{-x^2}\,dx = \sqrt{\pi}$ (fiche §4). Une machine ne peut pas aller jusqu'à l'infini : on calcule $I(B) = \int_{-B}^{B} e^{-x^2}\,dx$ pour des bornes $B$ de plus en plus grandes, avec `quad`. La cellule fournie définit la liste `bounds` et la valeur `gauss` $= \sqrt{\pi}$.

Calcule :
- a) $I(1)$ (4 décimales) ;
- b) le plus petit $B$ de `bounds` pour lequel $\lvert I(B) - \sqrt{\pi} \rvert < 10^{-6}$ (un entier) ;
- c) le rapport $\frac{I(3)}{\sqrt{\pi}}$ (6 décimales) ;
- d) $\int_{-6}^{6} e^{-x^2/2}\,dx$ (4 décimales), à comparer avec $\sqrt{2\pi}$.

Puis trace l'écart $\lvert I(B) - \sqrt{\pi} \rvert$ en fonction de $B$, de 0,5 à 5, avec une échelle **logarithmique** en ordonnée (`ax.semilogy`).

Dans ta copie : pour $\lvert x \rvert \ge B \ge 1$, $e^{-x^2} \le e^{-B\lvert x \rvert}$. Déduis-en une borne de l'écart $\sqrt{\pi} - I(B)$ (deux queues), et compare-la à ce que montre ta courbe. Pourquoi un ordinateur peut-il remplacer $+\infty$ par 6 pour la densité normale ?

In [ ]:
bounds = [1, 2, 3, 4, 5]
gauss = math.sqrt(math.pi)

In [ ]:
I_1 = ...          # a) a number
first_bound = ...  # b) an integer of bounds
ratio_3 = ...      # c) a number
normal_area = ...  # d) a number

In [ ]:
wbm.check("M02a.21a", I_1)
wbm.check("M02a.21b", first_bound)
wbm.check("M02a.21c", ratio_3)
wbm.check("M02a.21d", normal_area)

**Le tracé** de $\lvert I(B) - \sqrt{\pi} \rvert$ pour $B$ de 0,5 à 5 (`np.linspace`, `ax.semilogy`).

In [ ]:
# the plot of |I(B) − √π| against B, log scale on the y-axis

### Ex M02a.22 — L'aire qui se remplit : la courbe, les rectangles et la primitive 🎨 ★ ⏱️ 20 min
**Objectif :** dessiner une somme de Riemann, puis la fonction aire, et voir qu'elle coïncide avec une primitive.  
**Prérequis :** fiche §1 et §2 ; R0 (matplotlib) · **Parcours :** C, E

Complète deux fonctions, qui dessinent dans l'axe `ax` qu'elles reçoivent :
- `draw_riemann(ax, f, a, b, n)` : la courbe de $f$ sur $[a ; b]$ (`ax.plot`), les $n$ rectangles **au milieu** (`ax.bar(lefts, heights, width=h, align="edge", alpha=0.4)`, où `lefts` sont les abscisses $a + ih$ des bords gauches) et, dans le titre, la somme des aires des rectangles avec 4 décimales (`f"somme = {total:.4f}"`) ;
- `draw_area_function(ax, f, F, a, b)` : sur une grille de 200 points de $[a ; b]$, deux courbes : la **fonction aire** $A(x) = \int_a^x f$, calculée numériquement (méthode des trapèzes cumulée : `np.cumsum`), et $F(x) - F(a)$, calculée avec la primitive `F` ; avec une légende.

⚠️ `f` et `F` prennent **un** nombre (elles sont écrites avec `math`) : sur une grille `xs`, calcule `ys = np.array([f(x) for x in xs])`, pas `f(xs)`.

La cellule suivante les essaie sur $f(x) = 1 + x\,e^{-x}$ entre 0 et 4 (une primitive est $F(x) = x - (x + 1)e^{-x}$, par parties) ; une autre les vérifie sur une autre fonction.

Dans ta copie : les deux courbes de droite se superposent-elles ? Quel théorème cela illustre-t-il ? Où la fonction aire monte-t-elle le plus vite, et pourquoi là ?

In [ ]:
def draw_riemann(ax, f, a, b, n):
    """Draw f on [a, b], its n midpoint rectangles (ax.bar) and their total area in the title (4 decimals)."""
    raise NotImplementedError("ax.plot pour la courbe, ax.bar pour les rectangles, puis le titre avec la somme")


def draw_area_function(ax, f, F, a, b):
    """Draw the area function x -> ∫_a^x f (cumulative trapezoids) and F(x) − F(a) on 200 points of [a, b]."""
    raise NotImplementedError("une grille, np.cumsum des aires des trapèzes, puis les deux courbes et une légende")

In [ ]:
with checked("M02a.22 figure"):
    f_demo = lambda x: 1 + x * math.exp(-x)              # noqa: E731
    F_demo = lambda x: x - (x + 1) * math.exp(-x)        # noqa: E731
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
    try:
        created = drawn(draw_riemann, ax1, f_demo, 0, 4, 8) or drawn(draw_area_function, ax2, f_demo, F_demo, 0, 4)
    except BaseException:
        plt.close(fig)                                     # no empty figure while the functions are not written yet
        raise
    if created:
        plt.close(fig)
        print("❌ Ex M02a.22 : une de tes fonctions ouvre sa propre figure (plt.figure ou plt.subplots) : dessine "
              "dans l'axe qu'elle reçoit.")
    else:
        plt.show()

**Vérifications automatiques** (les figures elles-mêmes se jugent à l'œil).

In [ ]:
with checked("M02a.22 vérifications"):
    problems = []
    f_test, F_test = (lambda x: math.cos(x) + 1.5), (lambda x: math.sin(x) + 1.5 * x + 2)   # F(0) = 2, not 0
    fig, ax = plt.subplots(figsize=(5, 3))
    try:
        created = drawn(draw_riemann, ax, f_test, 0, 3, 6)
    finally:
        plt.close(fig)                                   # only the checks below are shown here
    if created:
        problems.append("draw_riemann ouvre sa propre figure : dessine dans l'axe reçu (ax.bar, ax.plot)")
    else:
        bars = [p for p in ax.patches if hasattr(p, "get_width")]
        mids = [0.25 + 0.5 * i for i in range(6)]
        if len(bars) != 6:
            problems.append(f"draw_riemann : n = 6 rectangles avec ax.bar (j'en compte {len(bars)})")
        else:
            widths = sorted(round(b.get_width(), 6) for b in bars)
            heights = sorted(round(b.get_height(), 6) for b in bars)
            lefts = sorted(round(b.get_x(), 6) for b in bars)
            if widths != [0.5] * 6 or lefts != [0.5 * i for i in range(6)]:
                problems.append("draw_riemann : des rectangles de largeur h = (b − a)/n, qui partent de a + i h "
                                "(ax.bar(..., width=h, align=\"edge\"))")
            elif heights != sorted(round(f_test(m), 6) for m in mids):
                problems.append("draw_riemann : la hauteur de chaque rectangle est f au milieu de son intervalle")
        if not ax.lines:
            problems.append("draw_riemann : la courbe de f, avec ax.plot")
        total = 0.5 * sum(f_test(m) for m in mids)
        if f"{total:.4f}" not in ax.get_title().replace(",", "."):
            problems.append("draw_riemann : la somme des aires des rectangles dans le titre, avec 4 décimales")
    fig, ax = plt.subplots(figsize=(5, 3))
    try:
        created = drawn(draw_area_function, ax, f_test, F_test, 0, 3)
    finally:
        plt.close(fig)
    if created:
        problems.append("draw_area_function ouvre sa propre figure : dessine dans l'axe reçu")
    else:
        exact = F_test(3) - F_test(0)
        curves = [line for line in ax.lines if len(np.atleast_1d(line.get_ydata())) >= 50]
        good = [line for line in curves if abs(float(np.atleast_1d(line.get_ydata())[0])) < 1e-9
                and abs(float(np.atleast_1d(line.get_ydata())[-1]) - exact) < 1e-3]
        if len(curves) < 2:
            problems.append("draw_area_function : deux courbes sur la grille de 200 points, l'aire calculée et "
                            "F(x) − F(a)")
        elif len(good) < 2:
            problems.append("draw_area_function : les deux courbes partent de 0 en x = a (une aire de a à a) et "
                            "finissent à F(b) − F(a), l'intégrale de a à b : trace F(x) − F(a), pas F(x)")
    if problems:
        print("❌ Ex M02a.22 : à revoir : " + " ; ".join(problems) + ".")
    else:
        print("✅ Ex M02a.22 : les rectangles, leur somme, et la fonction aire qui suit F(x) − F(a).")

## Partie C · Erreurs classiques, vérification et défi

Fiche §1, §2, §3 et §5. Trois erreurs à trouver, une primitive vérifiée par SymPy, puis le défi des dix évaluations.

### Ex M02a.23 — Chasse à l'erreur : bornes inversées, primitive fausse, aire négative 🐛 ★ ⏱️ 15 min
**Objectif :** repérer trois erreurs classiques du calcul intégral, puis calculer les bonnes valeurs.  
**Prérequis :** fiche §1 et §2 · **Parcours :** C

Voici cinq calculs, chacun avec ce qu'en dit son auteur. Trois sont faux. **Sans exécuter de code**, trouve-les.

- **(A)** « $\int_0^2 3x^2\,dx = \big[x^3\big]_2^0 = 0 - 8 = -8$. »
- **(B)** « Une primitive de $\frac{1}{x^2}$ est $-\frac{1}{x}$, donc $\int_1^2 \frac{dx}{x^2} = -\frac{1}{2} - (-1) = \frac{1}{2}$. »
- **(C)** « Une primitive de $e^{-3x}$ est $e^{-3x}$, donc $\int_0^1 e^{-3x}\,dx = e^{-3} - 1 \approx -0{,}9502$. »
- **(D)** « L'aire de la région comprise entre la courbe de $x^2 - 1$ et l'axe des abscisses, pour $x$ de 0 à 2, vaut $\int_0^2 (x^2 - 1)\,dx = \frac{8}{3} - 2 = \frac{2}{3}$. »
- **(E)** « $\int_1^e \frac{2}{x}\,dx = \big[2\ln x\big]_1^e = 2$. »

Questions :
- a) les lettres des calculs faux, dans l'ordre alphabétique (un texte) ;
- b) pour chacun des cinq, de (A) à (E), la bonne valeur de ce que décrit la phrase de son auteur (une liste de 5 nombres, 4 décimales).

Dans ta copie : pour chaque calcul faux, quelle règle a été oubliée ? Quel signe aurait dû alerter l'auteur de (C) ?

In [ ]:
answer_M02a_23a = ...  # a) letters, as text
answer_M02a_23b = ...  # b) a list of 5 values (4 decimals)

In [ ]:
wbm.check("M02a.23a", answer_M02a_23a)
wbm.check("M02a.23b", answer_M02a_23b)

### Ex M02a.24 — Vérifier une primitive avec SymPy : dériver le résultat 🛠️ ★ ⏱️ 10 min
**Objectif :** écrire une petite fonction qui vérifie une primitive en la dérivant, réflexe professionnel avant d'utiliser un résultat.  
**Prérequis :** fiche §2 · **Parcours :** C

Une primitive se vérifie toujours en la **dérivant** : c'est plus sûr que de la recalculer, et cela marche quelle que soit sa forme. Écris `is_primitive(F, f)`, qui reçoit deux expressions SymPy en `x` et renvoie `True` si $F' = f$, `False` sinon. Outils : `sp.diff(F, x)` dérive, `sp.simplify(expression)` simplifie ; teste si la simplification de $F' - f$ vaut 0, et renvoie un booléen de Python (`bool(...)`).

⚠️ Ne compare pas $F'$ et $f$ avec `==` sans simplifier : SymPy compare alors l'**écriture** des expressions, pas leurs valeurs, et $\sin x \cos x$ n'est pas écrit comme la dérivée de $-\frac{\cos(2x)}{4}$.

Dans ta copie : `sin²x/2` et `−cos(2x)/4` sont toutes deux des primitives de $\sin x \cos x$. Que vaut leur différence ? Est-ce une contradiction ?

In [ ]:
x = sp.Symbol("x")

In [ ]:
def is_primitive(F, f):
    """True when the SymPy expression F is an antiderivative of f in x (F' − f simplifies to 0)."""
    raise NotImplementedError("sp.diff, puis sp.simplify de la différence, comparée à 0")

In [ ]:
with checked("M02a.24 vérifications"):
    problems = []
    cases = [
        (sp.sin(x) ** 2 / 2, sp.sin(x) * sp.cos(x), True, "sin²x/2 est une primitive de sin x cos x"),
        (-sp.cos(2 * x) / 4, sp.sin(x) * sp.cos(x), True, "−cos(2x)/4 aussi (elle diffère de la précédente d'une "
                                                         "constante)"),
        (sp.exp(x ** 2), sp.exp(x ** 2), False, "e^(x²) n'est pas une primitive de e^(x²) (règle de la chaîne)"),
        (sp.atan(x), 1 / (1 + x ** 2), True, "arctan est une primitive de 1/(1 + x²)"),
        (x * sp.exp(x), x * sp.exp(x), False, "x eˣ n'est pas une primitive de x eˣ"),
    ]
    for F_case, f_case, expected, why in cases:
        result = is_primitive(F_case, f_case)
        if result is None:
            problems.append("is_primitive ne renvoie rien (None) : il manque un return")
            break
        if not isinstance(result, bool):
            problems.append(f"is_primitive doit renvoyer un booléen de Python, True ou False, et non un objet de type "
                            f"{type(result).__name__} : convertis avec bool(...)")
            break
        if result is not expected:
            problems.append(f"is_primitive se trompe : {why}")
    if problems:
        print("❌ Ex M02a.24 : à revoir : " + " ; ".join(problems) + ".")
    else:
        print("✅ Ex M02a.24 : is_primitive reconnaît une primitive, même écrite sous une autre forme.")

### Ex M02a.25 — Défi : la meilleure précision avec dix évaluations de la fonction 🏆 ★★★ ⏱️ 30 min
**Objectif :** calculer trois intégrales à $10^{-8}$ près en n'évaluant chaque fonction que dix fois au plus.  
**Prérequis :** fiche §3 et §5 ; Ex M02a.16 et M02a.17 · **Parcours :** C

Écris `best_integral(f, a, b)`, qui approche $\int_a^b f$ en appelant `f` **au plus 10 fois**. **Objectif** : une erreur inférieure à $10^{-8}$ sur chacune des trois intégrales de la cellule fournie (comparées à `quad`, très précis). La cellule de vérification compte les appels.

Simpson avec 8 intervalles (9 évaluations) donne des erreurs de l'ordre de $10^{-5}$ : loin du compte. Il faut une idée de plus. Pistes, de la plus simple à la plus efficace (`04_indices.md` les détaille) :
1. combiner deux résultats de Simpson, avec 4 et 8 intervalles, pour annuler le terme d'erreur principal (tous les points de l'un sont parmi ceux de l'autre) ;
2. renoncer aux points régulièrement espacés : il existe des points et des poids **choisis** pour être exacts sur des polynômes de degré bien plus élevé (cherche « quadrature de Gauss-Legendre » ; `np.polynomial.legendre.leggauss(n)` donne ces points et ces poids sur $[-1 ; 1]$, à ramener sur $[a ; b]$ par un changement de variable affine, §3).

Dans ta copie : quelle meilleure erreur obtiens-tu sur chaque intégrale, avec quelle méthode ? Pourquoi la troisième est-elle la plus difficile pour les méthodes à points réguliers ?

In [ ]:
CHALLENGE = [
    ("e^(−x²) sur [0 ; 2]", lambda x: math.exp(-x ** 2), 0, 2),
    ("1/(1 + x) sur [0 ; 1]", lambda x: 1 / (1 + x), 0, 1),
    ("√(1 + x³) sur [0 ; 2]", lambda x: math.sqrt(1 + x ** 3), 0, 2),
]
TARGET = 1e-8


def counted(f):
    """f, with a counter of its calls (counted_f.calls)."""
    def wrapper(x):
        wrapper.calls += 1
        return f(x)
    wrapper.calls = 0
    return wrapper

In [ ]:
def best_integral(f, a, b):
    """Approximate ∫_a^b f with at most 10 calls of f."""
    raise NotImplementedError("10 évaluations au plus : une combinaison de Simpson, ou des points de Gauss")

In [ ]:
with checked("M02a.25"):
    worst, calls_max = 0.0, 0
    for name, f, a, b in CHALLENGE:
        g = counted(f)
        value = best_integral(g, a, b)
        exact = quad(f, a, b, epsabs=1e-13, epsrel=1e-13)[0]
        worst, calls_max = max(worst, abs(value - exact)), max(calls_max, g.calls)
        print(f"{name:24} erreur : {abs(value - exact):.1e}   évaluations : {g.calls}")
    achieved = bool(worst < TARGET and calls_max <= 10)
    if calls_max > 10:
        print(f"❌ {calls_max} évaluations : 10 au plus.")
    wbm.check("M02a.25", achieved, computed=True)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Calcules-tu une intégrale avec une primitive, et sais-tu la distinguer d'une aire géométrique ?
2. Sais-tu programmer les rectangles, les trapèzes et Simpson, et prévoir leur erreur ?
3. Passes-tu d'une densité à des probabilités, à la fonction de répartition et aux quantiles ?

**La suite** : M02b, les variables aléatoires et les lois usuelles, puis M02c, avant le chapitre 2 de workbookIA.